In [1]:
import os
# Enforce single GPU to prevent DataParallel / multi-GPU synchronization conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import precision_recall_fscore_support
from huggingface_hub import login, hf_hub_download
import zipfile

In [ ]:
# ==========================================
# 1. AUTHENTICATION & DATA LOADING
# ==========================================
login(token=os.environ["HF_TOKEN"])

# Dataset repository from your Phase 4 setup
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO = "A-240242/legal-gqa-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_cjpe_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_4_CJPE/{filename}", 
        repo_type="dataset", 
        token=os.environ["HF_TOKEN"]
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 4 CJPE Data...")
train_data = load_cjpe_file("Train/train.json")
dev_data = load_cjpe_file("Train/dev.json")

print(f"Loaded {len(train_data)} training examples and {len(dev_data)} development examples.")

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 80.3MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Loaded 37387 training examples and 3505 development examples.


In [ ]:
# ==========================================
# 2. TOKENIZER & LONG-DOCUMENT SLIDING WINDOW CHUNKING
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=os.environ["HF_TOKEN"])
tokenizer.padding_side = "right"

def chunk_long_document(text, tokenizer, max_length=1024, stride=256):
    """
    Long-document handling technique: Sliding-window document chunking 
    to process extensive legal texts exceeding standard attention limits[cite: 5, 6].
    """
    tokenized = tokenizer(text, truncation=False, return_attention_mask=False)
    full_ids = tokenized["input_ids"]
    
    chunks = []
    step = max_length - stride
    
    if len(full_ids) <= max_length:
        return [full_ids]
        
    for i in range(0, len(full_ids), step):
        chunk = full_ids[i : i + max_length]
        chunks.append(chunk)
        if i + max_length >= len(full_ids):
            break
            
    return chunks

def prepare_dataset_records(raw_records, tokenizer, max_length=1024, stride=256):
    """
    Optimized long-document handling: Selects key strategic segments 
    (e.g., the beginning/facts and the end/ruling) instead of exhaustive chunk explosion.
    """
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    
    for record in raw_records:
        doc_text = record.get("text", "")
        label = record.get("label", 0)
        
        tokenized = tokenizer(doc_text, truncation=False, return_attention_mask=False)
        full_ids = tokenized["input_ids"]
        
        if len(full_ids) <= max_length:
            chunks = [full_ids]
        else:
            # Take the first chunk (facts/arguments) and the last chunk (rulings/conclusion)
            # This captures the dual anchors of legal judgments efficiently.
            first_chunk = full_ids[:max_length]
            last_chunk = full_ids[-max_length:]
            chunks = [first_chunk, last_chunk]
            
        for chunk in chunks:
            input_ids_list.append(chunk)
            attention_mask_list.append([1] * len(chunk))
            labels_list.append(label)
            
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "label": labels_list
    }

print("Processing Training and Development Datasets...")
train_features = prepare_dataset_records(train_data, tokenizer)
dev_features = prepare_dataset_records(dev_data, tokenizer)

train_dataset = Dataset.from_dict(train_features)
dev_dataset = Dataset.from_dict(dev_features)


[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Processing Training and Development Datasets...


In [17]:
# ==========================================
# 3. LORMA ADAPTER DEFINITION
# ==========================================
class LormaAdapter(nn.Module):
    def __init__(self, d_model, rank=16, alpha=32):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(d_model, rank, bias=False)
        self.B = nn.Linear(rank, d_model, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)
        
    def forward(self, x):
        return self.scaling * self.B(self.A(x))

In [18]:
import math
import torch
import torch.nn as nn
from transformers import LlamaConfig, LlamaForSequenceClassification
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

# ==========================================
# 4. GQA LORMA MODEL WRAPPER FOR CJPE (FIXED)
# ==========================================
class GQALormaForCJPE(nn.Module):
    def __init__(self, model_repo, num_labels=2, rank=16, alpha=32):
        super().__init__()
        
        # 1. Use LlamaConfig as the base configuration class for GQA compatibility
        self.config = LlamaConfig(
            vocab_size=len(tokenizer),
            hidden_size=768,
            num_hidden_layers=6,
            num_attention_heads=12,
            num_key_value_heads=4, # Configures Grouped-Query Attention (GQA)
            intermediate_size=2048,
            pad_token_id=tokenizer.pad_token_id,
            num_labels=num_labels
        )
        
        # 2. Initialize directly via LlamaForSequenceClassification using the config
        print("Initializing Base GQA Architecture...")
        self.base_model = LlamaForSequenceClassification(self.config)
        
        # 3. Fetch and load weights directly from safetensors
        print("Fetching GQA Checkpoint Weights from Hub...")
        model_weight_path = hf_hub_download(repo_id=model_repo, filename="model.safetensors", token=HF_TOKEN)
        state_dict = load_file(model_weight_path)
        
        # Load weights into base model (strict=False handles classification head mapping differences)
        self.base_model.load_state_dict(state_dict, strict=False)
        
        d_model = self.config.hidden_size
        
        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        # Unfreeze classification head
        for param in self.base_model.score.parameters():
            param.requires_grad = True
            
        # Inject LORMA adapters into linear projections
        self.lorma_adapters = nn.ModuleDict()
        for name, module in self.base_model.named_modules():
            if isinstance(module, nn.Linear) and ("self_attn" in name or "mlp" in name or "dense" in name):
                adapter_name = name.replace(".", "_")
                self.lorma_adapters[adapter_name] = LormaAdapter(d_model, rank=rank, alpha=alpha)
                for p in self.lorma_adapters[adapter_name].parameters():
                    p.requires_grad = True

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )
        return outputs

print("Initializing GQA LORMA Model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GQALormaForCJPE(HF_MODEL_REPO, num_labels=2, rank=16, alpha=32)

# Ensure trainable parameters are in FP32 for numerical stability
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready. Trainable Parameters: {trainable_params:,}")

Initializing GQA LORMA Model...
Initializing Base GQA Architecture...
Fetching GQA Checkpoint Weights from Hub...
Model ready. Trainable Parameters: 1,033,728


In [19]:
# ==========================================
# 5. METRICS & TRAINING CONFIGURATION
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = logits.argmax(axis=-1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./phase4_gqa_lorma_cjpe",
    report_to="none",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="macro_f1",
    load_best_model_at_end=True,
    fp16=True,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("\nCommencing GQA LORMA Training for Court Judgment Prediction (CJPE)...")
trainer.train()


Commencing GQA LORMA Training for Court Judgment Prediction (CJPE)...


Epoch,Training Loss,Validation Loss,Macro Precision,Macro Recall,Macro F1
1,No log,0.696590,0.491848,0.491667,0.491421
2,2.794031,0.690333,0.512500,0.512500,0.512500


TrainOutput(global_step=26, training_loss=2.72231780565702, metrics={'train_runtime': 14.8726, 'train_samples_per_second': 26.761, 'train_steps_per_second': 1.748, 'total_flos': 0.0, 'train_loss': 2.72231780565702, 'epoch': 2.0})

In [20]:
# ==========================================
# 6. INFERENCE & SUBMISSION GENERATION
# ==========================================
print("Loading Test Data for Submission Generation...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_4_CJPE/Test/test.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

model.eval()
predictions = []

for case in test_data:
    case_id = case.get("id")
    case_text = case.get("text", "")
    
    # Tokenize input test text
    inputs = tokenizer(
        case_text, 
        max_length=1024, 
        truncation=True, 
        return_tensors="pt"
    ).to(device)
    
    with torch.inference_mode():
        outputs = model(**inputs)
        logits = outputs.logits
        pred_label = int(logits.argmax(dim=-1).item())
        
    predictions.append({
        "id": case_id,
        "pred": pred_label
    })

# Save to expected submission format (`prediction.json` zipped)[cite: 9]
output_dir = "./phase4_gqa_lorma_cjpe"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nCJPE Prediction complete! Generated {len(predictions)} predictions[cite: 9].")
print(f"Submission saved to: {sub_zip_path}")

Loading Test Data for Submission Generation...


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            


CJPE Prediction complete! Generated 1517 predictions[cite: 9].
Submission saved to: ./phase4_gqa_lorma_cjpe/submission.zip
